# Chunk 2 — Market data + log returns

**Goal:** Pull ~10 years of SPY data and compute returns *correctly*.

**Focus concepts:**
- Simple return vs **log return** and why log wins for ML
- Adjusted close (dividends + splits)
- Stationarity — why we model returns, not prices

This chunk is short (~0.5 day). It sets up the dataset every subsequent chunk will use.

**Ground rules, same as last time:** fill the TODO cells, answer the Question cells in your head before continuing.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

## 2. Pull 10 years of SPY

We're pulling Jan 2015 through Dec 2025 — 11 full calendar years. That's ~2,700 trading days. Plenty for walk-forward validation with multiple ~1-year test windows.

**`auto_adjust=True` is important.** It adjusts the `Close` for dividends and stock splits. If you use raw close prices, your "return" on an ex-dividend day will look like a drop that didn't actually happen to an investor who held through it. Wrong returns → wrong everything.

In [ ]:
df = yf.download("SPY", start="2015-01-01", end="2025-12-31", auto_adjust=True)

# flatten MultiIndex columns (same trick as Chunk 1)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(df.shape)
print(df.index.min(), "→", df.index.max())
df.head()

**Question:** Roughly how many trading days did you get? Does that make sense for ~11 years? (Use 252 per year as a rule of thumb.)

## 3. Log returns

Two ways to measure a day's return:

- **Simple return:** $r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$ — what you got from `pct_change()` in Chunk 1.
- **Log return:** $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$.

We'll use **log returns** from here on. Why — in a minute. First compute them.

In [ ]:
# TODO: compute log returns using np.log and .shift(1)
# df["log_return"] = ...

# also keep simple returns for comparison
df["simple_return"] = df["Close"].pct_change()

df = df.dropna()
df[["Close", "log_return", "simple_return"]].head()

## 4. Simple vs log — are they different?

Plot them against each other. They should lie almost perfectly on the line `y = x`.

**Why?** For small `x`, `ln(1 + x) ≈ x`. Daily equity returns are small (typically < 2% in magnitude), so log ≈ simple. The difference only becomes material for large moves.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["simple_return"], df["log_return"], alpha=0.3, s=8)
lims = [df["simple_return"].min(), df["simple_return"].max()]
ax.plot(lims, lims, "r--", label="y = x")
ax.set_xlabel("Simple return")
ax.set_ylabel("Log return")
ax.set_title("Log vs Simple return — daily SPY")
ax.legend()
plt.show()

**Question:** Look at the scatter plot. Where do the two returns diverge most — near zero, or at the tails? Which one is always smaller in magnitude for large positive returns, and why? (Hint: `ln(1.05) vs 0.05`, `ln(1.20) vs 0.20`.)

## 5. Why we actually use log returns: they're *additive*

Here's the real reason log returns win. Consider the total return over N days:

- **Simple returns compound multiplicatively:**
  $$\text{Total}_{simple} = (1 + r_1)(1 + r_2)\cdots(1 + r_N) - 1$$
- **Log returns compound additively:**
  $$\text{Total}_{log} = r_1 + r_2 + \cdots + r_N$$

Additive is much easier to work with: you can take rolling sums, compute moments (mean, variance), feed to most ML models. Multiplicative compounding breaks all of that.

**Let's verify.** Compute two things:
1. `sum_log` — the sum of all log returns
2. `log_total_ratio` — `ln(last_price / first_price)`

They should be identical (up to floating point).

In [ ]:
first_price = df["Close"].iloc[0]
last_price = df["Close"].iloc[-1]

# TODO: compute sum_log (sum of all log returns) and log_total_ratio (log of last/first)
# sum_log = ...
# log_total_ratio = ...

# print both and check they're close
# print(f"sum of log returns: {sum_log:.6f}")
# print(f"log(P_end / P_start): {log_total_ratio:.6f}")
# print(f"np.isclose? {np.isclose(sum_log, log_total_ratio)}")

**Question:** If SPY's closing price went from ~$200 (Jan 2015) to ~$550 (end of 2025), what does `log(550/200)` evaluate to approximately? What does that number *mean*? (Is it a percentage? A ratio?)

Write down your guess, then let the numbers print above and see if you were close.

## 6. Stationarity — why we model returns, not prices

**Stationary** (loosely): the statistical properties of the series — mean, variance, distribution shape — don't change over time.

Most ML models implicitly assume this: they fit on training data and expect test data to come from the same distribution. If your training data has SPY at $200 and your test data has SPY at $500, a model trained on raw prices will get wrecked — it's never seen a $500 before.

**Returns are approximately stationary.** Prices are not. Look at the two plots below and notice:
- Price chart: mean clearly drifts upward over time (non-stationary)
- Return chart: bounces around zero, roughly constant volatility — *mostly* stationary (there are volatility regime changes, e.g. COVID 2020)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(df.index, df["Close"])
ax1.set_title("SPY Close — non-stationary (mean drifts up)")
ax1.set_ylabel("Price ($)")

ax2.plot(df.index, df["log_return"], linewidth=0.5)
ax2.set_title("SPY log return — approximately stationary (mean ~ 0)")
ax2.set_ylabel("Log return")
ax2.axhline(0, color="red", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.show()

**Question:** On the log-return plot, can you visually spot the COVID crash (Mar 2020)? What specifically changed about the series around that time — the *mean*, the *variance*, or both? This matters later: regime changes are real, and a model trained only on calm periods may fail in volatile ones.

## 7. Return distribution — a sanity check

Plot the histogram of log returns. You should see:
- A fat peak near zero (most days, nothing much happens)
- Tails that extend further than a normal distribution would predict ("fat tails" — famous empirical fact about financial returns)
- Approximately symmetric around zero, slightly negatively skewed (crashes are sharper than rallies)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["log_return"], bins=100, edgecolor="black", alpha=0.75)
ax.axvline(df["log_return"].mean(), color="red", linestyle="--", label=f"mean = {df['log_return'].mean():.5f}")
ax.set_xlabel("Log return")
ax.set_ylabel("Count")
ax.set_title("Distribution of daily SPY log returns")
ax.legend()
plt.show()

print(f"Mean daily log return: {df['log_return'].mean():.5f}  (~= {df['log_return'].mean() * 252:.3f} annualized)")
print(f"Std  daily log return: {df['log_return'].std():.5f}  (~= {df['log_return'].std() * np.sqrt(252):.3f} annualized)")
print(f"Skew:                  {df['log_return'].skew():.3f}")
print(f"Kurtosis (excess):     {df['log_return'].kurtosis():.3f}  (normal = 0; positive = fatter tails)")

**Question:** The annualized mean is roughly the "average return per year" if you held SPY. The annualized std is roughly the annual volatility. Does the ratio (mean/std) look familiar? (Hint: it's the Sharpe ratio of buy-and-hold.)

**Question:** Kurtosis > 0 means fatter tails than a normal distribution. In plain English, what does "fat tails" mean for a risk model? Why does your new team care about this specifically?

## Done-when checklist

- [ ] You have a DataFrame `df` with a `log_return` column, no NaNs, ~2,700 rows, spanning Jan 2015 – Dec 2025.
- [ ] You can articulate *why* we use log returns (two reasons: ≈ simple returns for small moves, and they're additive).
- [ ] You can articulate *why* we model returns, not prices (stationarity).
- [ ] You've seen the fat-tail shape of the return distribution and know that matters for risk modeling.

Ping me when you've worked through it. Chunk 3 is where we build features — this is where the plan spends real time, so we'll slow down.